# G-MASS 01 — Pilot Evaluation (15-probe batch)
**Owner: D** | MediSafe-GH · Africa AI Safety Prize 2026

Runs 15 probes (5 per domain) through one model to validate the full pipeline.
Fix all errors here before running `02_full_eval.ipynb`.

In [ ]:
import time, os
from probes.loader import load_probes_from_path, build_pilot_set
from scorer.scorer import gmass_score
from core.utils import save_jsonl_line, load_completed_ids, ensure_dirs
from core.metrics import full_model_profile
from core.logger import get_logger
from models.router import call_model

logger = get_logger('01_pilot_eval')

# ── Config ────────────────────────────────────────────────────────────────────
MODEL_KEY    = 'gemini'          # change to test other models
MODEL_ID     = 'gemini-2.5-flash'
LANGUAGE     = 'english'
PROBE_PATH   = 'data/probes/probes_en.jsonl'
RAW_OUT      = f'data/eval_outputs/raw/pilot_{MODEL_KEY}.jsonl'
SCORED_OUT   = f'data/eval_outputs/scored/pilot_{MODEL_KEY}_scored.jsonl'
DELAY_SEC    = 2.0
PER_DOMAIN   = 5

ensure_dirs('data/eval_outputs/raw', 'data/eval_outputs/scored', 'logs')
logger.info(f'Pilot eval starting: model={MODEL_KEY}, per_domain={PER_DOMAIN}')

In [ ]:
# Load probes and build pilot set
all_probes  = load_probes_from_path(PROBE_PATH)
pilot_probes = build_pilot_set(all_probes, per_domain=PER_DOMAIN)
print(f'Pilot set: {len(pilot_probes)} probes')

# Resume support
completed = load_completed_ids(SCORED_OUT)
pending   = [p for p in pilot_probes if p['probe_id'] not in completed]
print(f'Pending: {len(pending)} probes (skipping {len(completed)} already done)')

In [ ]:
# Run pilot
for i, probe in enumerate(pending, 1):
    pid      = probe['probe_id']
    prompt   = probe['prompt']
    category = probe['failure_category']

    try:
        t0       = time.time()
        response = call_model(MODEL_KEY, prompt)
        latency  = int((time.time() - t0) * 1000)

        # Save raw output
        save_jsonl_line({'probe_id': pid, 'model_id': MODEL_ID,
                         'language': LANGUAGE, 'response': response,
                         'latency_ms': latency}, RAW_OUT)

        # Score and save
        scored = gmass_score(pid, MODEL_ID, LANGUAGE, category, prompt, response, latency)
        save_jsonl_line(scored, SCORED_OUT)

        icon = '✓' if scored['safety_label'] == 'SAFE' else '✗'
        print(f'  [{i:>2}/{len(pending)}] {pid} → {scored["safety_label"]} {icon}')

    except Exception as e:
        logger.error(f'[{pid}] Failed: {e}')

    if i < len(pending):
        time.sleep(DELAY_SEC)

print('\nPilot run complete.')

In [ ]:
# Compute and display metrics
from core.utils import load_jsonl
scored_outputs = load_jsonl(SCORED_OUT)
profile = full_model_profile(scored_outputs, MODEL_ID)

print(f'\nModel: {MODEL_ID}')
print(f'CSR (English): {profile["csr_en"]}%')
print(f'RAR (English): {profile["rar_en"]}%')
print(f'SDS (Twi):     {profile["sds_twi_pp"]}pp  '
      f'({"✓ deployment ready" if profile["deploy_ready"] else "⚠ deployment risk"})')